In [2]:
from google.colab import files

uploaded = files.upload()

Saving student_data.csv to student_data.csv


In [ ]:
# ==============================
# IMPORTS
# ==============================
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


# ==============================%
# LOAD DATA
# ==============================
df = pd.read_csv("student_data.csv")

X = df[["task_hours", "days_left", "confidence", "stress"]]
y = df["risk"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression()
model.fit(X_train, y_train)

print("Model Accuracy:", accuracy_score(y_test, model.predict(X_test)))


# ==============================
# RULE SYSTEM
# ==============================
def rule_system(task_hours, days_left, confidence, stress):
    explanations = []

    if days_left <= 3 and task_hours >= 5:
        explanations.append("You have a large task with very little time remaining")

    if stress == 3:
        explanations.append("Your stress level is high, which may affect productivity")

    if confidence == 1:
        explanations.append("Low confidence suggests difficulty in completing tasks")

    if days_left > 5 and task_hours <= 3:
        explanations.append("You have enough time and manageable workload")

    # FINAL DECISION
    if len(explanations) >= 2:
        risk = "HIGH"
    elif len(explanations) == 1:
        risk = "MEDIUM"
    else:
        risk = "LOW"

    return risk, explanations


# ==============================
# BFS PLANNER
# ==============================
def planner_bfs(tasks, available_hours):
    schedule = {}
    hours_copy = available_hours.copy()

    for name, hours, deadline in tasks:
        for day in hours_copy:
            if hours_copy[day] > 0 and hours > 0:
                study = min(hours, hours_copy[day])

                if day not in schedule:
                    schedule[day] = []

                schedule[day].append((name, study))
                hours_copy[day] -= study
                hours -= study

    return schedule


# ==============================
# GREEDY PLANNER (deadline + size)
# ==============================
def planner_greedy(tasks, available_hours):
    schedule = {}
    hours_copy = available_hours.copy()

    tasks_sorted = sorted(tasks, key=lambda x: (x[2], -x[1]))

    for name, hours, deadline in tasks_sorted:
        for day in hours_copy:
            if hours_copy[day] > 0 and hours > 0:
                study = min(hours, hours_copy[day])

                if day not in schedule:
                    schedule[day] = []

                schedule[day].append((name, study))
                hours_copy[day] -= study
                hours -= study

    return schedule


# ==============================
# MAIN SYSTEM
# ==============================
print("\n===== STUDENT SUCCESS COPILOT =====")

# ---- TASK INPUT
n = int(input("Enter number of tasks: "))

tasks = []
total_task_hours = 0
deadlines = []

for i in range(n):
    name = input(f"Enter name of task {i+1}: ")
    hours = int(input(f"Enter hours for {name}: "))
    deadline = int(input(f"Days left until deadline for {name}: "))

    tasks.append((name, hours, deadline))
    total_task_hours += hours
    deadlines.append(deadline)

# ---- TIME INPUT
available_hours = {}
days = int(input("Enter number of study days: "))

for i in range(days):
    day_name = input(f"Enter day {i+1} name: ")
    hrs = int(input(f"Available hours on {day_name}: "))
    available_hours[day_name] = hrs

# ---- OTHER INPUT
confidence = int(input("Confidence (1-3): "))
stress = int(input("Stress (1-3): "))

# ==============================
# BACKWARD CHAINING (ADDED PART)
# ==============================
if confidence not in [1, 2, 3]:
    print("System: I need a valid confidence level (1-3).")
    confidence = int(input("Enter confidence (1-3): "))

if stress not in [1, 2, 3]:
    print("System: I need a valid stress level (1-3).")
    stress = int(input("Enter stress (1-3): "))

# ---- VALIDATION
total_available = sum(available_hours.values())
days_left = min(deadlines)

if total_available < total_task_hours:
    print("\n⚠ Warning: Not enough study time to finish all tasks!")

if any(d <= 0 for d in deadlines):
    print("\n⚠ Warning: Invalid deadline detected!")


# ==============================
# PLANNER OUTPUT
# ==============================
schedule = planner_greedy(tasks, available_hours)

print("\n===== STUDY PLAN =====")
for day, work in schedule.items():
    print(day, ":", work)


# ==============================
# SEARCH COMPARISON
# ==============================
bfs_schedule = planner_bfs(tasks, available_hours)

print("\n=== SEARCH COMPARISON ===")
print("BFS allocations:", sum(len(v) for v in bfs_schedule.values()))
print("Greedy allocations:", sum(len(v) for v in schedule.values()))
print("Greedy prioritizes urgent and large tasks.")


# ==============================
# ML + RULES
# ==============================
ml_risk = "HIGH" if model.predict([[total_task_hours, days_left, confidence, stress]])[0] == 1 else "LOW"

rule_risk, explanations = rule_system(total_task_hours, days_left, confidence, stress)

# FINAL DECISION
if ml_risk == "HIGH" or rule_risk == "HIGH":
    final_risk = "HIGH"
elif rule_risk == "MEDIUM":
    final_risk = "MEDIUM"
else:
    final_risk = "LOW"


# ==============================
# FINAL OUTPUT
# ==============================
print("\n===== RESULTS =====")
print("ML Risk:", ml_risk)
print("Rule Risk:", rule_risk)
print("Final Risk:", final_risk)

print("\nDetailed Explanation:")

if len(explanations) == 0:
    print("- You are in a safe situation with enough time and manageable workload.")
else:
    for e in explanations:
        print("-", e)

Model Accuracy: 1.0

===== STUDENT SUCCESS COPILOT =====
